In [ ]:
import os
import pandas as pd
import boto3

try:
    import kagglehub
except:
    ! pip install kagglehub
    import kagglehub

try:
    import pyarrow
except:
    ! pip install pyarrow
    import pyarrow

## Helper Classes

In [ ]:
class DataCollector:
    def __init__(self, str_bucket):
        self.str_bucket = str_bucket
        self.s3_client = boto3.client('s3')
        self.df_data = None

    def download_from_kaggle(self):
        print('Downloading Groceries dataset from Kaggle...')
        str_path = kagglehub.dataset_download('heeraldedhia/groceries-dataset')
        print(f'Dataset downloaded to: {str_path}')
        list_files = [f for f in os.listdir(str_path) if f.endswith('.csv')]
        if not list_files:
            raise FileNotFoundError('No CSV file found in downloaded dataset')
        str_csv_path = os.path.join(str_path, list_files[0])
        print(f'Loading CSV: {str_csv_path}')
        self.df_data = pd.read_csv(str_csv_path)
        return self.df_data

    def clean_data(self):
        if self.df_data is None:
            raise ValueError('Data not loaded. Call download_from_kaggle() first.')
        print('\nCleaning data...')
        # standardize column names
        self.df_data.columns = [c.strip().lower().replace(' ', '_') for c in self.df_data.columns]
        # rename for consistency
        dict_rename = {
            'member_number': 'member_id',
            'date': 'transaction_date',
            'itemdescription': 'item'
        }
        self.df_data = self.df_data.rename(columns=dict_rename)
        # parse date
        self.df_data['transaction_date'] = pd.to_datetime(self.df_data['transaction_date'], dayfirst=True)
        # strip whitespace from item names and lowercase
        self.df_data['item'] = self.df_data['item'].str.strip().str.lower()
        # drop nulls
        int_nulls = self.df_data.isnull().sum().sum()
        print(f'Total null values: {int_nulls}')
        self.df_data = self.df_data.dropna()
        # ensure member_id is int
        self.df_data['member_id'] = self.df_data['member_id'].astype('int64')
        # sort by date
        self.df_data = self.df_data.sort_values('transaction_date').reset_index(drop=True)
        print(f'Data shape after cleaning: {self.df_data.shape}')
        return self.df_data

    def get_summary_stats(self):
        if self.df_data is None:
            raise ValueError('Data not loaded.')
        print('\n=== SUMMARY STATISTICS ===')
        print(f'Total rows (item-level): {len(self.df_data):,}')
        print(f'Unique members: {self.df_data["member_id"].nunique():,}')
        print(f'Unique items: {self.df_data["item"].nunique():,}')
        print(f'Date range: {self.df_data["transaction_date"].min()} to {self.df_data["transaction_date"].max()}')
        # transactions = unique (member, date) pairs
        int_transactions = self.df_data.groupby(['member_id', 'transaction_date']).ngroups
        print(f'Total transactions (member + date): {int_transactions:,}')
        flt_avg_basket = len(self.df_data) / int_transactions
        print(f'Average items per transaction: {flt_avg_basket:.2f}')
        print(f'\nTop 10 items:')
        print(self.df_data['item'].value_counts().head(10))
        print(f'\nColumns: {list(self.df_data.columns)}')

    def save_to_s3(self, str_filename):
        if self.df_data is None:
            raise ValueError('Data not loaded.')
        str_s3_key = f'00_data_collection/{str_filename}'
        print(f'\nSaving to S3: s3://{self.str_bucket}/{str_s3_key}')
        from io import BytesIO
        buffer = BytesIO()
        self.df_data.to_parquet(buffer, index=False)
        buffer.seek(0)
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print(f'Successfully uploaded {str_filename} to S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')

## Constants

In [ ]:
str_bucket = 'market-basket-analysis-demo'
str_task = 'data_collection'

## Download and Prepare Data

In [ ]:
cls_collector = DataCollector(str_bucket)
df_raw = cls_collector.download_from_kaggle()

In [ ]:
print('First few rows of raw data:')
print(df_raw.head(10))
print(f'\nData shape: {df_raw.shape}')
print(f'\nColumn names and types:')
print(df_raw.dtypes)

In [ ]:
df_cleaned = cls_collector.clean_data()

In [ ]:
cls_collector.get_summary_stats()

In [ ]:
cls_collector.save_to_s3('groceries.parquet')

## Completion

In [ ]:
print('\n=== DATA COLLECTION COMPLETE ===')
print(f'File ready for next stage: s3://{str_bucket}/00_data_collection/groceries.parquet')